<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-10e-synthetic-data-generation-filtering-and-measuring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Discussion 10 (continued) - Synthetic data: generate, filter, decontaminate, and measure whether it helped

Meridian Bank has 77 customer-message intents and almost no labelled examples for the new ones. A model can write plausible extra examples, quickly and for free. The question this lab answers is the one that matters: **does training on those examples help, and what does it take to make them help?** We generate synthetic training data with an open-weight model, run it through a filtering pipeline (parse, deduplicate, check label consistency, remove anything that duplicates the test set), and measure the effect on a held-out test set, with the controls that stop you fooling yourself.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded; not needed for the certificate. Runs on a CPU (a GPU runtime is used automatically and is faster); the generation step takes a few minutes on a laptop-class CPU and longer on a free Colab CPU.

*Original code written for this course. Data: the Banking77 customer-intent dataset (PolyAI, CC BY 4.0; Casanueva et al., 2020), loaded from a public copy. Generator: Qwen2.5-1.5B-Instruct (Apache-2.0). Embeddings: all-MiniLM-L6-v2. The numbers below come from one seed set; Part 5 says how much to trust them.*

In [ ]:
%pip install -q transformers accelerate sentence-transformers scikit-learn pandas pyarrow numpy

In [ ]:
import re, time, random, collections, itertools
import numpy as np, pandas as pd, torch
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
SEED = 0
rng = np.random.default_rng(SEED); random.seed(SEED); torch.manual_seed(SEED)
BASE = "https://huggingface.co/datasets/mteb/banking77/resolve/main/data/"
train = pd.read_parquet(BASE + "train-00000-of-00001.parquet"); test = pd.read_parquet(BASE + "test-00000-of-00001.parquet")
print(len(train), "train and", len(test), "test messages;", train.label_text.nunique(), "intents")

## Part 1 - The low-label setting

We pick 15 intents at random, then simulate scarcity: only **5 real labelled examples per intent** (75 in all) are available for training. The official test messages for those 15 intents are the yardstick, and they stay untouched until the evaluation. The classifier is deliberately simple (sentence embeddings plus logistic regression) so that differences come from the *data*, not from clever modelling.

In [ ]:
INTENTS = sorted(rng.choice(train.label_text.unique(), 15, replace=False))
tr = train[train.label_text.isin(INTENTS)].reset_index(drop=True); te = test[test.label_text.isin(INTENTS)].reset_index(drop=True)
seed_idx = tr.groupby("label_text").sample(n=5, random_state=SEED).index
SEEDS = tr.loc[seed_idx].reset_index(drop=True); POOL = tr.drop(seed_idx).reset_index(drop=True)      # POOL: the real data we pretend not to have
print(len(SEEDS), "real seed messages |", len(POOL), "real messages we are pretending not to have |", len(te), "test messages")
print(INTENTS)
enc = SentenceTransformer("all-MiniLM-L6-v2")
def emb(texts): return enc.encode(list(texts), batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
E_test = emb(te.text)
def fit_eval(texts, labels):
    clf = LogisticRegression(max_iter=2000, C=10).fit(emb(texts), labels)
    pred = clf.predict(E_test); correct = (pred == te.label_text.to_numpy())
    return correct.mean(), correct, clf
acc_seed, c_seed, clf_seed = fit_eval(SEEDS.text, SEEDS.label_text)
print(f"trained on the 75 real seeds only: test accuracy {acc_seed:.3f}")

## Part 2 - Generate

For each real seed message we ask the model for four *different* messages from a customer with the same intent, sampled at temperature 0.8 so the four differ. The prompt gives the intent's name and the seed. Raw generations are noisy; keeping them raw is the first mistake to measure.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
GEN_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(GEN_ID, padding_side="left")
try: gen = AutoModelForCausalLM.from_pretrained(GEN_ID, dtype=torch.float32).eval()
except TypeError: gen = AutoModelForCausalLM.from_pretrained(GEN_ID, torch_dtype=torch.float32).eval()
dev = "cuda" if torch.cuda.is_available() else "cpu"; gen.to(dev)

PROMPT = ("You write realistic short messages that customers send to a bank's support chat.\n"
          "Intent: {intent}\nExample message: \"{seed}\"\n"
          "Write 4 different messages from other customers with the same intent. Vary the wording and the situation. "
          "One message per line, no numbering, no quotation marks.")
def generate(batch):
    msgs = [tok.apply_chat_template([{"role": "user", "content": PROMPT.format(intent=r.label_text.replace('_', ' '), seed=r.text)}], add_generation_prompt=True, tokenize=False) for r in batch.itertuples()]
    enc_in = tok(msgs, return_tensors="pt", padding=True).to(dev)
    with torch.no_grad():
        out = gen.generate(**enc_in, max_new_tokens=110, do_sample=True, temperature=0.8, top_p=0.95)
    return [tok.decode(o[enc_in["input_ids"].shape[1]:], skip_special_tokens=True) for o in out]

t0 = time.time(); RAW = []
for s in range(0, len(SEEDS), 15):
    batch = SEEDS.iloc[s:s + 15]
    for r, text in zip(batch.itertuples(), generate(batch)):
        RAW.append({"intent": r.label_text, "seed": r.text, "output": text})
    print(f"  generated {min(s + 15, len(SEEDS))}/{len(SEEDS)} seeds, {time.time() - t0:.0f}s", end="\r")
print("\nexample raw output for intent", RAW[0]["intent"], "->", repr(RAW[0]["output"][:300]))

## Part 3 - The filtering pipeline

Four stages, each with a count so you can see what it removed:

1. **Parse and clean.** Split into lines, strip list markers, bold markers and quotes; drop lines that are too short, too long, not ending in a full stop, question mark or exclamation mark (a line cut off by the token limit), or copies of the seed.
2. **Deduplicate.** Drop a message if its character 5-gram Jaccard similarity to an already kept message (real seeds included) is 0.8 or more.
3. **Label consistency.** Keep a message only if a classifier trained on the *real seeds alone* assigns it the intended intent. (This is a self-consistency filter: it cannot add knowledge the seeds lack, and it tends to discard the hard, boundary-near examples; Part 4 shows what that costs.)
4. **Decontaminate.** Drop any message that is a near-duplicate (Jaccard 0.8 or more) of a *test* message. A model that has seen the public dataset may reproduce test items; training on them would inflate the score for the wrong reason.

In [ ]:
def clean_lines(text):
    out = []
    for ln in text.split("\n"):
        ln = re.sub(r"^\s*(?:[-*\u2022]|\d+[.)])\s*", "", ln).replace("**", "").strip().strip('"\u201c\u201d').strip()
        complete = ln.endswith((".", "?", "!"))                          # a line cut off by the token limit is dropped
        if complete and 3 <= len(ln.split()) <= 40 and not ln.lower().startswith(("intent", "example", "here are", "sure")): out.append(ln)
    return out

def shingles(s, k=5):
    s = re.sub(r"\s+", " ", s.lower()).strip(); return {s[i:i + k] for i in range(max(1, len(s) - k + 1))}
def jacc(a, b): return len(a & b) / len(a | b) if a | b else 0.0

cand = [{"intent": r["intent"], "text": t, "seed": r["seed"]} for r in RAW for t in clean_lines(r["output"]) if t.lower() != r["seed"].lower()]
stages = collections.OrderedDict([("generated lines after parse/clean", len(cand))])

kept, kept_sh = [], [shingles(t) for t in SEEDS.text]
for c in cand:
    sh = shingles(c["text"])
    if all(jacc(sh, k) < 0.8 for k in kept_sh): kept.append(c); kept_sh.append(sh)
stages["after deduplication"] = len(kept)

E_c = emb([c["text"] for c in kept]); pr = clf_seed.predict_proba(E_c); cls = list(clf_seed.classes_)
consistent = [c for c, p in zip(kept, pr) if cls[int(p.argmax())] == c["intent"]]
stages["after label-consistency filter"] = len(consistent)

test_sh = [shingles(t) for t in te.text]
leaks = [c for c in kept if any(jacc(shingles(c["text"]), t) >= 0.8 for t in test_sh)]
clean_final = [c for c in consistent if all(jacc(shingles(c["text"]), t) < 0.8 for t in test_sh)]
stages["after decontamination (final)"] = len(clean_final)
for k, v in stages.items(): print(f"{k:38}{v:>6}")
print(f"\nnear-duplicates of TEST messages among the deduplicated generations: {len(leaks)}")
print("final examples per intent:", dict(collections.Counter(c['intent'] for c in clean_final)))

## Part 4 - Did it help? Four conditions and two controls

We train the same classifier on different data and always score on the same held-out test messages:

- **seeds only** (75 real messages): the starting point;
- **seeds + raw synthetic**: everything that parsed (stages 1 only, no dedup, no consistency filter);
- **seeds + filtered synthetic**: the full pipeline;
- **seeds + the same number of extra *real* messages** (control: what the synthetic data is worth in real labels);
- **a much larger real set** (control: the ceiling, using the real data we pretended not to have).

Intervals are bootstrap 95% over the test messages.

In [ ]:
def ci(correct, B=2000, seed=1):
    r = np.random.default_rng(seed); n = len(correct); b = [correct[r.integers(0, n, n)].mean() for _ in range(B)]
    return np.percentile(b, 2.5), np.percentile(b, 97.5)

def mix(extra):
    return list(SEEDS.text) + [c["text"] for c in extra], list(SEEDS.label_text) + [c["intent"] for c in extra]

raw_only = [c for c in cand]                                         # no dedup, no consistency filter
n_extra = len(clean_final)
real_extra = POOL.sample(n=n_extra, random_state=SEED)
big_real = POOL.groupby("label_text").sample(n=50, random_state=SEED)

conds = collections.OrderedDict()
conds["seeds only (75 real)"] = fit_eval(SEEDS.text, SEEDS.label_text)
t, l = mix(raw_only);       conds[f"seeds + raw synthetic ({len(raw_only)})"] = fit_eval(t, l)
t, l = mix(clean_final);    conds[f"seeds + filtered synthetic ({n_extra})"] = fit_eval(t, l)
conds[f"seeds + {n_extra} extra REAL (control)"] = fit_eval(list(SEEDS.text) + list(real_extra.text), list(SEEDS.label_text) + list(real_extra.label_text))
conds["seeds + 750 REAL (ceiling)"] = fit_eval(list(SEEDS.text) + list(big_real.text), list(SEEDS.label_text) + list(big_real.label_text))
def paired_ci(c1, c0, B=2000, seed=2):
    r = np.random.default_rng(seed); d = c1.astype(float) - c0.astype(float); n = len(d)
    b = [d[r.integers(0, n, n)].mean() for _ in range(B)]; return d.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)
c0 = conds["seeds only (75 real)"][1]
print(f"{'training data':44}{'test accuracy':>15}   95% interval      change vs seeds only (paired 95% interval)")
for k, (a, c, _) in conds.items():
    lo, hi = ci(c)
    if c is c0: print(f"{k:44}{a:>15.3f}   [{lo:.3f}, {hi:.3f}]   -")
    else:
        d, dl, dh = paired_ci(c, c0); print(f"{k:44}{a:>15.3f}   [{lo:.3f}, {hi:.3f}]   {d:+.3f} [{dl:+.3f}, {dh:+.3f}]")

In [ ]:
# What did the consistency filter keep and drop? Compare the generated sets on diversity and distance from the seed.
def distinct2(texts, k=75, draws=30):
    # share of distinct word pairs, on random subsets of the SAME size (the measure falls as a set grows, so sizes must match)
    r = random.Random(0); vals = []
    for _ in range(draws):
        sub = r.sample(texts, min(k, len(texts)))
        bigr = [b for t in sub for b in zip(t.lower().split(), t.lower().split()[1:])]; vals.append(len(set(bigr)) / max(1, len(bigr)))
    return float(np.mean(vals))
def mean_len(texts): return np.mean([len(t.split()) for t in texts])
sets = {"real seeds": list(SEEDS.text), "real test messages": list(te.text), "synthetic, all parsed": [c["text"] for c in raw_only], "synthetic, final": [c["text"] for c in clean_final]}
print(f"{'set':26}{'n':>6}{'mean words':>12}{'distinct-2 (75 sampled)':>25}")
for k, v in sets.items(): print(f"{k:26}{len(v):>6}{mean_len(v):>12.1f}{distinct2(v):>25.3f}")
dropped = [c for c in kept if c not in consistent]
print("\nexamples the consistency filter DROPPED (intended intent, text):")
for c in random.Random(0).sample(dropped, min(5, len(dropped))): print("  ", c["intent"], "|", c["text"])
print("\nexamples it KEPT:")
for c in random.Random(1).sample(consistent, min(5, len(consistent))): print("  ", c["intent"], "|", c["text"])

**What one run showed.** The generator produced 266 usable messages from 75 seeds (the pipeline then kept 210); no message was dropped as a near-duplicate and none duplicated a test message. Seeds only scored 0.896. Adding the raw synthetic messages gave +0.015 (paired 95% interval -0.005 to +0.035), adding the filtered set +0.017 (0.000 to +0.033): a small gain that the intervals cannot cleanly separate from zero, and no measurable difference between raw and filtered. The same *number* of extra real labelled messages gave +0.045 (+0.025 to +0.067), and 750 real ones +0.060. So in this run the synthetic set was worth something, but a real label was worth well over twice as much as a synthetic one.

**How to read the results.** Look at the rows in this order.

1. *Seeds only* is the floor.
2. *Raw synthetic* versus *filtered synthetic*: the effect of the pipeline. Check whether raw data helped, did nothing, or hurt, and whether filtering changed it. Do not assume the answer; it depends on the generator, the task and the prompt.
3. *Extra real* is the honest price tag: if the same number of real labels would have done much better, the synthetic set was worth that gap in labels, not more.
4. The *ceiling* says how much headroom there was in the first place.
5. The distinct-2 and length rows show whether the synthetic text is as varied as real text. Here the word-pair diversity was about the same as real text (0.78 against 0.79 and 0.75) while the synthetic messages were much longer (18.8 words against 11.7), so the generator was wordier, not more repetitive; the measure cannot tell you whether the *situations* described vary. Training repeatedly on a model's own output narrows a distribution over generations, the mechanism behind "model collapse" in the research literature, so watch diversity if you ever feed synthetic data back into the generator.
6. The dropped and kept examples are the reason to *read your data*. Among those dropped, some are mislabelled by the generator (a message about paying off a credit card under 'pending card payment') and some are perfectly good but hard (a message about adding funds with a debit card under 'topping up by card'): a consistency filter removes errors and also the boundary-near cases a classifier most needs. Among those kept, some are still off-intent. No automatic filter replaces reading a sample.

## Part 5 - How much to trust this

- **One seed set, one generation run.** The intervals above reflect test-set sampling only, not the randomness of which five messages were chosen as seeds or of the sampled generations. A difference of a point or two between conditions is within that noise. To make a claim, repeat with several seed draws and report the spread; the notebook's structure makes that a loop around Parts 1-4.
- **A self-consistency filter and its limits.** It cannot correct a systematic error the seeds already contain.
- **Contamination is checked by near-duplicate text only.** Paraphrases of test items would pass that check.
- **Generator licence and terms.** The generator here is Apache-2.0, so its outputs carry no extra restriction. Many hosted model providers restrict using their outputs to train competing models; read the terms before generating a training set from one.
- **Where this fits.** Synthetic data is most dependable when you can *verify* it mechanically (code that runs, arithmetic that checks, a schema that validates), as in the verifier-based reinforcement learning of Chapter 11. For free-text labels like these, the filter is a proxy for verification and the measured effect is the only evidence.

## Part 6 - Documenting a synthetic dataset

If you keep and share the set, record: the generator and its revision, the prompt, the sampling settings and seed, the seed data and its licence, every filter with its counts (the table in Part 3), what was checked for contamination, the evaluation above, and the known limitations. A one-page datasheet next to the file is enough.

## What to hand in

The Part 4 table for your own 15-intent draw (change the seed), one paragraph on whether filtering helped *in your run* with the interval, and one example of a synthetic message that passed every filter and is still wrong or unrealistic.